# Crop Yantra — Yield and Rice-Health Training
R Ritwik (23BAI0037) · Joshith G (23BAI0075)

Upload the private transfer ZIP prepared on your PC. Select **Runtime → Change runtime type → GPU** for image training. Both training switches are **False** by default; running all cells therefore prepares and checks data without fitting project models.

| Pipeline | Model | Inputs | Train / validation / test |
|---|---|---|---|
| Yield | MLP and HistGradientBoostingRegressor × standard/DAE | 28 standard or 16 DAE predictors | 1,485 / 245 / 288 aligned observations |
| Rice health | ImageNet-pretrained EfficientNet-B0 | Four rice-photo classes | 1,120 / 240 / 240 photos |

The DAE is already fitted. No preprocessing is refitted here. The neighbour-augmented table is not substituted. Keep this ZIP private; it contains third-party images subject to source terms.

## 1. Upload and verify the transfer bundle
Choose `crop_yantra_training.zip`. This cell rejects unsafe paths, duplicate members and mismatched hashes. Use only your trusted locally built ZIP: a hash manifest is integrity checking, not independent proof of authorship. Extraction refuses an existing project directory.

In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, json, stat, zipfile
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one training ZIP.")
archive = Path(next(iter(uploaded)))
del uploaded
ROOT = Path("/content/crop_yantra_training")
if ROOT.exists():
    raise FileExistsError("Project directory exists. Reuse it in the following cells; do not extract again.")
with zipfile.ZipFile(archive) as bundle:
    members = bundle.infolist()
    names = [item.filename for item in members]
    if len(names) != len(set(names)):
        raise ValueError("Duplicate archive member.")
    if sum(item.file_size for item in members) > 1024**3:
        raise ValueError("Unexpected uncompressed transfer size.")
    for item in members:
        path = PurePosixPath(item.filename)
        if path.is_absolute() or ".." in path.parts or "\\" in item.filename or ":" in item.filename:
            raise ValueError("Unsafe archive path.")
        if stat.S_ISLNK(item.external_attr >> 16) or item.is_dir():
            raise ValueError("Only regular file members are supported.")
    transfer = json.loads(bundle.read("TRANSFER_MANIFEST.json"))
    expected = transfer["files"]
    if set(names) != set(expected) | {"TRANSFER_MANIFEST.json"}:
        raise ValueError("Unlisted transfer members.")
    for name, digest in expected.items():
        if hashlib.sha256(bundle.read(name)).hexdigest() != digest:
            raise ValueError(f"Transfer hash mismatch: {name}")
    ROOT.mkdir()
    for name in expected:
        target = ROOT / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(bundle.read(name))
print(f"Verified and extracted {len(expected)} files. No credentials or raw archive required.")

## 2. Install shared processing libraries
Keep Colab's matched CUDA-enabled torch/torchvision pair; do not install the Windows CPU lock. Dependencies below match the saved tabular preprocessing. A new subprocess is used for checks and training so it sees newly installed packages. If installation or preflight fails, resolve that error before enabling training.

In [ ]:
from pathlib import Path
import subprocess, sys
ROOT = Path("/content/crop_yantra_training")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "APP/training/requirements.txt")], check=True)
subprocess.run([sys.executable, "-c", "import torch,torchvision; print('torch:',torch.__version__,'torchvision:',torchvision.__version__,'CUDA:',torch.cuda.is_available())"], check=True)
print("Training has not started.")

## 3. Check both pipelines
Checks every selected photo, split separation, class mapping, model input/output shapes, saved preprocessing compatibility and both yield representations. This does not fit estimators or update neural weights. CUDA is used by the image runner when available; the small yield experiments run on CPU.

In [ ]:
subprocess.run([sys.executable, "APP/training/run.py"], cwd=ROOT, check=True)
readiness = json.loads((ROOT / "APP/training/readiness.json").read_text())
assert readiness["status"] == "ready"
print("Both pipelines passed. No supervised training started.")

## 4. Explicit training controls
Change only the switches for the tasks you intend to run. Yield performs four experiments. Image training starts with 5 head-only epochs, then fine-tunes the last two backbone blocks, up to 15 total epochs. Validation chooses checkpoints; test data is evaluated after selection, not used for tuning. Training duration and accuracy are not guaranteed.

In [ ]:
TRAIN_YIELD = False
TRAIN_IMAGES = False
SEED = 42
YIELD_EPOCHS = 250
IMAGE_EPOCHS = 15
IMAGE_HEAD_EPOCHS = 5
IMAGE_BATCH_SIZE = 32

if TRAIN_YIELD:
    subprocess.run([sys.executable, "APP/training/run.py", "--task", "yield", "--train",
                    "--yield-epochs", str(YIELD_EPOCHS), "--seed", str(SEED)], cwd=ROOT, check=True)
else:
    print("Yield training skipped.")
if TRAIN_IMAGES:
    subprocess.run([sys.executable, "APP/training/run.py", "--task", "images", "--train",
                    "--image-epochs", str(IMAGE_EPOCHS), "--image-head-epochs", str(IMAGE_HEAD_EPOCHS),
                    "--image-batch-size", str(IMAGE_BATCH_SIZE), "--seed", str(SEED)], cwd=ROOT, check=True)
else:
    print("Image training skipped.")

## 5. Inspect and download results
Yield outputs include MAE/RMSE/R², validation-ranked comparison, training-mean baseline and held-out predictions. Image outputs include accuracy, macro-F1, per-class metrics, confusion matrix and history. Preserve results before the temporary Colab runtime ends. No model is activated in the webpage. Do not claim South-India-wide accuracy or combine image diagnoses with yield loss without appropriate paired validation.

In [ ]:
import shutil
runs = ROOT / "APP/training/runs"
if runs.exists() and any(runs.iterdir()):
    for name in sorted(runs.rglob("*.json")):
        print(name.relative_to(runs))
        if name.name in ("comparison.json", "metrics.json"):
            result = json.loads(name.read_text())
            print(json.dumps({key: value for key, value in result.items() if key != "history"}, indent=2))
    result_zip = shutil.make_archive("/content/crop_yantra_results", "zip", runs)
    files.download(result_zip)
else:
    print("No training results yet. Enable an explicit training switch only when intended.")